---
## Step 0: Import All Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.preprocessing import PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             precision_score, recall_score, f1_score)

import joblib

---
## Step 1: Load the Dataset

In [2]:
df = pd.read_csv('students.csv')

In [63]:
df.head(10)

,Study_Hours_Per_Day,Attendance_%,Previous_GPA,Assignments_Submitted,Sleep_Hours,Final_Marks
0,4.4,90.4,3.85,1.0,6.1,1
1,9.6,94.8,3.88,2.0,8.5,1
2,NaN,65.9,NaN,0.0,4.6,1
3,6.4,55.5,2.43,0.0,NaN,1
4,99.0,NaN,1.54,2.0,NaN,0
5,2.4,71.4,3.82,4.0,6.3,1
6,1.5,90.9,9.90,2.0,4.3,0
7,8.8,93.0,3.92,0.0,4.6,1
8,NaN,50.3,3.91,0.0,4.6,1
9,7.4,75.5,3.63,7.0,7.2,1


---
## Step 2: Exploratory Data Analysis (EDA)

In [7]:
df.dtypes

Unnamed: 0                 int64
Student_ID                object
Student_Name              object
Study_Hours_Per_Day      float64
Attendance_%             float64
Previous_GPA             float64
Assignments_Submitted     object
Sleep_Hours               object
Final_Marks                int64
dtype: object

In [8]:
# Summary statistics
df.describe()

,Unnamed: 0,Study_Hours_Per_Day,Attendance_%,Previous_GPA,Final_Marks
count,125.000000,114.000000,114.000000,115.000000,125.000000
mean,62.000000,6.160526,75.514912,2.848870,0.864000
std,36.228442,9.401399,21.434045,1.069368,0.344168
min,0.000000,-3.000000,-10.000000,-1.000000,0.000000
25%,31.000000,2.800000,62.025000,2.235000,1.000000
50%,62.000000,5.300000,75.300000,2.870000,1.000000
75%,93.000000,7.900000,86.375000,3.370000,1.000000
max,124.000000,99.000000,200.000000,9.900000,1.000000


In [13]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2)
})
missing_df[missing_df['Missing Count'] > 0]

,Missing Count,Missing %
Study_Hours_Per_Day,11,8.8
Attendance_%,11,8.8
Previous_GPA,10,8.0
Sleep_Hours,10,8.0


In [85]:
duplicates = df.duplicated().sum()
df = df.drop_duplicates()

### 2.4 — Target Variable Distribution (Pass vs Fail)

In [41]:
# Count of Pass vs Fail
df['Final_Marks'].value_counts()
df['Final_Marks'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%'

Final_Marks
1    86.4%
0    13.6%
Name: proportion, dtype: object

---
## Step 3: Data Cleaning & Preprocessing

### 3.1 — Drop Unnecessary Columns

In [92]:
df = df.drop(columns=['Unnamed: 0', 'Student_ID', 'Student_Name'])

KeyError: "['Unnamed: 0', 'Student_ID', 'Student_Name'] not found in axis"

### 3.3 — Define Features (X) and Target (y)

In [65]:
# X = all features  |  y = target variable
X = df.drop(columns=['Final_Marks'])
y = df['Final_Marks']
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(y.value_counts())


X shape: (125, 5)
y shape: (125,)
Final_Marks
1    108
0     17
Name: count, dtype: int64


---
## Step 4: Feature Engineering & Column Transformer

In [67]:
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

---
## Step 5: Train-Test Split

In [95]:
# Split: 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.20,random_state=42)

### 4.2 — Build the ColumnTransformer (Preprocessing Pipeline)

In [96]:
numerical_pipeline = Pipeline(steps=[
    ('imputer',    SimpleImputer(strategy='median')),       # Fill NaN
    ('transformer', PowerTransformer(method='yeo-johnson')), # Fix skewness
    ('scaler',     StandardScaler())                        # Scale features
])

# ColumnTransformer applies pipeline to selected columns
preprocessor = ColumnTransformer(transformers=[('num', numerical_pipeline, numerical_cols)])

In [70]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
}


In [84]:
results = {}
trained_pipelines = {}
for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier',   model)
    ])

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Calculate metrics
    acc       = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)
    f1        = f1_score(y_test, y_pred)
    roc_auc   = roc_auc_score(y_test, y_prob)

    results[model_name] = {
        'Accuracy' : round(acc, 4),
        'Precision': round(precision, 4),
        'Recall'   : round(recall, 4),
        'F1-Score' : round(f1, 4),
        'ROC-AUC'  : round(roc_auc, 4)
    }

    trained_pipelines[model_name] = pipeline

    print(f"✅ {model_name:22s} | Accuracy: {acc:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")

✅ Logistic Regression    | Accuracy: 0.8400 | F1: 0.9091 | ROC-AUC: 0.6364


In [76]:
# Save the best model pipeline using joblib
model_filename = 'best_student_model.pkl'

joblib.dump(best_pipeline, model_filename)

['best_student_model.pkl']

In [77]:
# Load the saved model
loaded_model = joblib.load('best_student_model.pkl')
y_pred_loaded = loaded_model.predict(X_test)
loaded_acc    = accuracy_score(y_test, y_pred_loaded)


In [94]:
new_student = pd.DataFrame({
    'Study_Hours_Per_Day': [0],
    'Attendance_%': [0],
    'Previous_GPA': [2.1],
    'Assignments_Submitted': [0],
    'Sleep_Hours': [18]
})

# Predict result
prediction = loaded_model.predict(new_student)

# Show result
if prediction[0] == 1:
    print("Student will PASS")
else:
    print("Student will FAIL")

Student will FAIL
